In [ ]:
# -*- coding: utf-8 -*-
"""
Compare Linear Regression, Random Forest, and LSTM
for ECOSTRESS-based river water temperature prediction.

Models:
    - Linear Regression (LR, with StandardScaler)
    - Random Forest (RF)
    - LSTM (sequence-based, with per-fold StandardScaler)

Target:
    T_w_obs  (in-situ river water temperature, °C)

Predictors (same across all models):
    - ECOSTRESS Ts: center_mean, buf50m_mean, ..., buf500m_mean
    - Discharge Q
    - Air temperature T_air
    - Static catchment attributes: DEM, slope, LULC fractions (500 m buffer)
    - Time/season features: doy_sin, doy_cos, hour
    - Monthly & seasonal Q and T_air summaries

Data:
    Clean table:
        C:\goyal_shekhar\post_doc\ecostress\ecostress_master_training_clean.csv

Outputs:
    Figures and CSVs in:
        C:\goyal_shekhar\post_doc\ecostress\figure_models\compare_LR_RF_LSTM\
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sklearn models
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Keras / TensorFlow for LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# ---------------------------------------------------------------------
# Paths and constants
# ---------------------------------------------------------------------
MASTER_CSV = Path(
    r"C:\goyal_shekhar\post_doc\ecostress\ecostress_master_training_clean.csv"
)
OUT_DIR = Path(
    r"C:\goyal_shekhar\post_doc\ecostress\figure_models\compare_LR_RF_LSTM"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 5  # sequence length for LSTM

# For reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ---------------------------------------------------------------------
# Load and prepare data
# ---------------------------------------------------------------------
print(f"Loading clean master training table from:\n  {MASTER_CSV}")
df = pd.read_csv(MASTER_CSV, parse_dates=["time"])
print("Raw training table shape:", df.shape)

# Fill short-buffer NaNs with center_mean (as before)
for r in [50, 100, 150]:
    col = f"buf{r}m_mean"
    if col in df.columns:
        before = df[col].isna().sum()
        df[col] = df[col].fillna(df["center_mean"])
        after = df[col].isna().sum()
        print(f"{col}: filled {before - after} NaNs with center_mean")

# Feature set (same as RF & LSTM scripts)
buffer_cols = [
    "buf50m_mean",
    "buf100m_mean",
    "buf150m_mean",
    "buf200m_mean",
    "buf300m_mean",
    "buf400m_mean",
    "buf500m_mean",
]

feature_cols = [
    "center_mean",
    *buffer_cols,
    "Q",
    "T_air",
    "DEM_m",
    "slope_deg_or_pct",
    "lulc_10",
    "lulc_30",
    "lulc_40",
    "lulc_50",
    "lulc_60",
    "doy_sin",
    "doy_cos",
    "hour",
    "Q_month_mean",
    "Q_month_max",
    "T_month_mean",
    "T_month_max",
    "Q_season_mean",
    "Q_season_max",
    "T_season_mean",
    "T_season_max",
]

target_col = "T_w_obs"

missing_features = [c for c in feature_cols if c not in df.columns]
if missing_features:
    raise RuntimeError(f"Missing features in table: {missing_features}")

# Drop rows with NaNs in predictors or target
mask_valid = (~df[feature_cols].isna().any(axis=1)) & (~df[target_col].isna())
df = df.loc[mask_valid].reset_index(drop=True)
print("Rows kept after NaN filter:", len(df))
print("Stations in filtered table:", df["station_name"].unique())

# Tabular data for LR & RF (& Lasso if you want it later)
X_tab = df[feature_cols].values
y_tab = df[target_col].values
stations_tab = df["station_name"].values
time_tab = df["time"].values

print("Tabular modelling data shape:", X_tab.shape, "| target len:", len(y_tab))

# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------
def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}


def stationwise_metrics_from_predictions(df_eval, y_pred, label="model"):
    """
    df_eval must contain columns: station_name, T_w_obs
    y_pred: array aligned with df_eval
    Returns a DataFrame with station-wise R2/RMSE/MAE.
    """
    df_tmp = df_eval.copy()
    df_tmp[f"T_w_pred_{label}"] = y_pred

    stats = []
    for st, g in df_tmp.groupby("station_name"):
        y_true = g["T_w_obs"].values
        y_hat  = g[f"T_w_pred_{label}"].values
        if len(g) < 20 or np.all(np.isnan(y_hat)):
            continue

        r2_st   = r2_score(y_true, y_hat)
        rmse_st = np.sqrt(mean_squared_error(y_true, y_hat))
        mae_st  = mean_absolute_error(y_true, y_hat)

        stats.append(
            {
                "station_name": st,
                "n": len(g),
                "R2": r2_st,
                "RMSE": rmse_st,
                "MAE": mae_st,
            }
        )

    return pd.DataFrame(stats).sort_values("R2", ascending=False)


# ---------------------------------------------------------------------
# A) Linear Regression and Random Forest (5-fold CV, sample-wise)
# ---------------------------------------------------------------------
models_sklearn = {
    # LR with in-fold scaling
    "Linear Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("regressor", LinearRegression()),
        ]
    ),
    # If you want LASSO as well, uncomment this:
    # "Lasso": Pipeline(
    #     steps=[
    #         ("scaler", StandardScaler()),
    #         ("regressor", Lasso(alpha=0.01, max_iter=10000, random_state=42)),
    #     ]
    # ),
    "Random Forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42,
    ),
}

kf_tab = KFold(n_splits=5, shuffle=True, random_state=42)

global_metrics_tab = {}            # global metrics per model
cv_predictions_tab = {}            # y_pred_cv per model
station_metrics_tab = {}           # station-wise metrics per model

print("\n" + "=" * 60)
print("A) SKLEARN MODELS — 5-FOLD CV (sample-wise)")
print("=" * 60)

for model_name, base_model in models_sklearn.items():
    print(f"\n=== 5-fold CV for {model_name} ===")
    y_pred_cv = np.full_like(y_tab, np.nan, dtype=float)
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(kf_tab.split(X_tab), start=1):
        X_train, X_test = X_tab[train_idx], X_tab[test_idx]
        y_train, y_test = y_tab[train_idx], y_tab[test_idx]

        model = clone(base_model)
        model.fit(X_train, y_train)
        y_hat = model.predict(X_test)

        y_pred_cv[test_idx] = y_hat

        mets = compute_metrics(y_test, y_hat)
        fold_metrics.append(mets)
        print(
            f"  Fold {fold}: R²={mets['R2']:.3f}, "
            f"RMSE={mets['RMSE']:.3f}, MAE={mets['MAE']:.3f}"
        )

    mets_df = pd.DataFrame(fold_metrics)
    global_metrics_tab[model_name] = mets_df.mean().to_dict()
    cv_predictions_tab[model_name] = y_pred_cv

    # station-wise metrics
    st_df = stationwise_metrics_from_predictions(
        df_eval=df[["station_name", "T_w_obs"]].copy(),
        y_pred=y_pred_cv,
        label=model_name.replace(" ", "_"),
    )
    station_metrics_tab[model_name] = st_df

    print(f"\n=== Station-wise 5-fold CV performance ({model_name}) ===")
    print(st_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    # save station-wise metrics
    st_out = OUT_DIR / f"{model_name.replace(' ', '_')}_station_metrics_tabCV.csv"
    st_df.to_csv(st_out, index=False)
    print(f"  Saved: {st_out}")


# ---------------------------------------------------------------------
# B) LSTM (sequence-based 5-fold CV, with scaling)
# ---------------------------------------------------------------------
def build_sequences_for_station(df_st, feature_cols, target_col, seq_len):
    df_st = df_st.sort_values("time").reset_index(drop=True)
    X_list, y_list, t_list, st_list = [], [], [], []

    if len(df_st) < seq_len:
        return None, None, None, None

    for i in range(len(df_st) - seq_len + 1):
        window = df_st.iloc[i : i + seq_len]
        X_list.append(window[feature_cols].values)
        y_list.append(window[target_col].values[-1])
        t_list.append(window["time"].values[-1])
        st_list.append(window["station_name"].values[-1])

    return (
        np.array(X_list),
        np.array(y_list),
        np.array(t_list),
        np.array(st_list),
    )


def build_lstm_model(n_features, seq_len=SEQ_LEN, units=64, dropout=0.2):
    model = Sequential()
    # Use explicit Input to avoid warning
    model.add(tf.keras.Input(shape=(seq_len, n_features)))
    model.add(LSTM(units, return_sequences=False))
    if dropout > 0:
        model.add(Dropout(dropout))
    model.add(Dense(32, activation="relu"))
    model.add(Dense(1, activation="linear"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"],
    )
    return model


print("\n" + "=" * 60)
print("B) LSTM — 5-FOLD CV (sequence-based, with scaling)")
print("=" * 60)

stations_all = df["station_name"].unique()
X_seq_list, y_seq_list, time_seq_list, st_seq_list = [], [], [], []

print(f"Building sequences with SEQ_LEN = {SEQ_LEN} …")
for st in stations_all:
    df_st = df[df["station_name"] == st].copy()
    X_s, y_s, t_s, st_s = build_sequences_for_station(
        df_st, feature_cols, target_col, SEQ_LEN
    )
    if X_s is None:
        print(f"  [skip sequences] {st}: only {len(df_st)} samples (< {SEQ_LEN})")
        continue
    X_seq_list.append(X_s)
    y_seq_list.append(y_s)
    time_seq_list.append(t_s)
    st_seq_list.append(st_s)
    print(f"  {st}: built {len(y_s)} sequences")

X_seq = np.concatenate(X_seq_list, axis=0)
y_seq = np.concatenate(y_seq_list, axis=0)
time_seq = np.concatenate(time_seq_list, axis=0)
station_seq = np.concatenate(st_seq_list, axis=0)

print("X_seq shape:", X_seq.shape, "(n_samples, seq_len, n_features)")
print("y_seq shape:", y_seq.shape)
print("Stations used in sequences:", np.unique(station_seq))

kf_seq = KFold(n_splits=5, shuffle=True, random_state=42)
n_features = X_seq.shape[2]

y_pred_lstm_cv = np.full_like(y_seq, np.nan, dtype=float)
fold_metrics_lstm = []

for fold, (train_idx, test_idx) in enumerate(kf_seq.split(X_seq), start=1):
    print(f"\n--- LSTM Fold {fold}/{kf_seq.n_splits} ---")
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y_seq[train_idx], y_seq[test_idx]

    # --- feature scaling (no leakage) ---
    scaler = StandardScaler()
    X_train_flat = X_train.reshape(-1, n_features)
    scaler.fit(X_train_flat)
    X_train_scaled = scaler.transform(X_train_flat).reshape(X_train.shape)
    X_test_scaled = scaler.transform(
        X_test.reshape(-1, n_features)
    ).reshape(X_test.shape)

    model_lstm = build_lstm_model(n_features=n_features, seq_len=SEQ_LEN)

    es = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    )

    history = model_lstm.fit(
        X_train_scaled,
        y_train,
        epochs=200,
        batch_size=32,
        validation_split=0.1,
        callbacks=[es],
        verbose=0,
    )

    y_hat = model_lstm.predict(X_test_scaled, verbose=0).ravel()
    y_pred_lstm_cv[test_idx] = y_hat

    mets = compute_metrics(y_test, y_hat)
    fold_metrics_lstm.append(mets)
    print(
        f"  Fold {fold}: R²={mets['R2']:.3f}, "
        f"RMSE={mets['RMSE']:.3f}, MAE={mets['MAE']:.3f}"
    )

mets_lstm_df = pd.DataFrame(fold_metrics_lstm)
global_metrics_lstm = mets_lstm_df.mean().to_dict()

print("\n=== LSTM global 5-fold CV performance (sequence-based) ===")
print(f"R²   : {global_metrics_lstm['R2']:.3f}")
print(f"RMSE : {global_metrics_lstm['RMSE']:.3f} °C")
print(f"MAE  : {global_metrics_lstm['MAE']:.3f} °C")

# Station-wise LSTM metrics
lstm_stats = []
for st in np.unique(station_seq):
    mask_st = (station_seq == st)
    y_true_st = y_seq[mask_st]
    y_hat_st  = y_pred_lstm_cv[mask_st]
    if np.sum(mask_st) < 20 or np.all(np.isnan(y_hat_st)):
        continue

    r2_st   = r2_score(y_true_st, y_hat_st)
    rmse_st = np.sqrt(mean_squared_error(y_true_st, y_hat_st))
    mae_st  = mean_absolute_error(y_true_st, y_hat_st)

    lstm_stats.append(
        {
            "station_name": st,
            "n_seq": np.sum(mask_st),
            "R2": r2_st,
            "RMSE": rmse_st,
            "MAE": mae_st,
        }
    )

station_metrics_lstm = pd.DataFrame(lstm_stats).sort_values("R2", ascending=False)
print("\n=== Station-wise LSTM performance (sequence-based CV) ===")
print(station_metrics_lstm.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

station_metrics_lstm.to_csv(
    OUT_DIR / "LSTM_station_metrics_seqCV.csv", index=False
)

# ---------------------------------------------------------------------
# C) Global comparison: LR vs RF vs LSTM
# ---------------------------------------------------------------------
print("\n" + "=" * 60)
print("C) Global comparison: LR vs RF vs LSTM")
print("=" * 60)

# Combine global metrics into one DataFrame
global_compare = pd.DataFrame.from_dict(
    {
        "Linear Regression": global_metrics_tab["Linear Regression"],
        "Random Forest": global_metrics_tab["Random Forest"],
        "LSTM": global_metrics_lstm,
    },
    orient="index",
)[["R2", "RMSE", "MAE"]]

print("\n=== Global 5-fold CV comparison ===")
print(global_compare.to_string(float_format=lambda x: f"{x:.3f}"))

global_compare.to_csv(OUT_DIR / "global_CV_metrics_LR_RF_LSTM.csv")

# ---------------------------------------------------------------------
# D) Plots: global metrics + obs vs pred comparison
# ---------------------------------------------------------------------
print("\nCreating comparison plots …")

# 1) Barplot of global metrics
plt.figure(figsize=(10, 4))
for i, metric in enumerate(["R2", "RMSE", "MAE"], start=1):
    ax = plt.subplot(1, 3, i)
    global_compare[metric].plot(kind="bar", ax=ax)
    if metric == "R2":
        ax.set_ylabel(r"Cross-validated $R^2$")
    else:
        ax.set_ylabel(metric + " (°C)")
    ax.set_title(metric)
    ax.set_xticklabels(global_compare.index, rotation=45, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.suptitle("Five-fold cross-validated performance: LR vs RF vs LSTM")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(OUT_DIR / "global_metrics_LR_RF_LSTM.png", dpi=300)
plt.show()

# 2) Obs vs Pred scatter for all three models
#    LR & RF use tabular samples; LSTM uses sequence targets
y_lr = y_tab
y_rf = y_tab
y_lstm = y_seq

y_pred_lr = cv_predictions_tab["Linear Regression"]
y_pred_rf = cv_predictions_tab["Random Forest"]
y_pred_lstm = y_pred_lstm_cv

# Determine common axis limits across all
all_obs = np.concatenate([y_lr, y_lstm])
all_pred = np.concatenate(
    [
        y_pred_lr[~np.isnan(y_pred_lr)],
        y_pred_rf[~np.isnan(y_pred_rf)],
        y_pred_lstm[~np.isnan(y_pred_lstm)],
    ]
)
vmin = min(all_obs.min(), all_pred.min())
vmax = max(all_obs.max(), all_pred.max())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes = axes.ravel()

model_plot_info = [
    ("Linear Regression", y_lr, y_pred_lr),
    ("Random Forest", y_rf, y_pred_rf),
    ("LSTM", y_lstm, y_pred_lstm),
]

for ax, (name, y_true_m, y_pred_m) in zip(axes, model_plot_info):
    valid = ~np.isnan(y_pred_m)
    y_true_v = y_true_m[valid]
    y_pred_v = y_pred_m[valid]

    ax.scatter(y_true_v, y_pred_v, s=8, alpha=0.4)
    ax.plot([vmin, vmax], [vmin, vmax], "k--", linewidth=1)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_xlabel(r"Observed $T_\mathrm{w}$ (°C)")
    ax.set_ylabel(r"Predicted $\hat{T}_\mathrm{w}$ (°C)")
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

fig.suptitle("Observed vs predicted river water temperature (5-fold CV)", y=0.95)
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(OUT_DIR / "obs_vs_pred_LR_RF_LSTM.png", dpi=300)
plt.show()

print("\n✅ Done. Global metrics, station-wise metrics, and comparison plots saved in:")
print(f"  {OUT_DIR}")
